In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

spark = (
    SparkSession.builder
    .appName("Complete_DataFrame_Transformations_Actions")
    .master("local[*]")   
    .getOrCreate()
)

spark

In [2]:
employee_data = [
    (101, "Anuj",   "IT",      90000, 29, "Mumbai",    1001, "2023-01-10", 4.7, None),
    (102, "Riya",   "HR",      60000, 31, "Pune",      1002, "2022-11-05", 4.2, "A"),
    (103, "Vikas",  "IT",      85000, 27, "Bengaluru", 1001, "2024-02-12", 4.5, "B"),
    (104, "Sneha",  "Finance", 95000, 35, "Mumbai",    1003, "2021-08-19", 4.8, "A"),
    (105, "Amit",   "Sales",   50000, 26, "Delhi",     1004, "2023-06-01", 3.9, None),
    (106, "Pooja",  "HR",      65000, 30, "Chennai",   1002, "2020-12-11", 4.1, "B"),
    (107, "Karan",  "IT",     120000, 33, "Hyderabad", 1001, "2019-03-14", 4.9, "A"),
    (108, "Meera",  "Sales",   52000, 28, "Pune",      1004, "2024-01-18", 3.7, "C"),
    (109, "Rohit",  "Finance", 88000, 32, "Delhi",     1003, "2022-09-09", 4.0, "B"),
    (110, "Nisha",  "Ops",     70000, 29, "Mumbai",    1005, "2023-04-22", 4.3, None),
]

employee_schema = StructType([
    StructField("emp_id", IntegerType(), False),
    StructField("emp_name", StringType(), False),
    StructField("dept", StringType(), True),
    StructField("salary", IntegerType(), True),
    StructField("age", IntegerType(), True),
    StructField("city", StringType(), True),
    StructField("manager_id", IntegerType(), True),
    StructField("join_date", StringType(), True),
    StructField("rating", DoubleType(), True),
    StructField("grade", StringType(), True),
])

emp_df = spark.createDataFrame(employee_data, employee_schema) \
    .withColumn("join_date", F.to_date("join_date"))

dept_data = [
    ("IT", "Technology", "North"),
    ("HR", "Human Resources", "West"),
    ("Finance", "Finance & Accounts", "North"),
    ("Sales", "Business Sales", "South"),
    ("Ops", "Operations", "West"),
    ("Legal", "Legal Affairs", "East"),
]

dept_df = spark.createDataFrame(dept_data, ["dept", "dept_full_name", "region"])

bonus_data = [
    (101, 15000, "2024"),
    (103, 12000, "2024"),
    (104, 20000, "2024"),
    (107, 30000, "2024"),
    (111, 10000, "2024"),
]
bonus_df = spark.createDataFrame(bonus_data, ["emp_id", "bonus_amount", "bonus_year"])

nested_data = [
    (1, ["python", "spark", "aws"], {"laptop": "dell", "level": "senior"}, ("A", 10)),
    (2, ["sql", "powerbi"], {"laptop": "hp", "level": "mid"}, ("B", 20)),
    (3, [], {"laptop": "mac", "level": "lead"}, ("C", 30)),
]
nested_schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("skills", ArrayType(StringType()), True),
    StructField("profile_map", MapType(StringType(), StringType()), True),
    StructField("code_struct", StructType([
        StructField("code", StringType(), True),
        StructField("score", IntegerType(), True),
    ]), True),
])
nested_df = spark.createDataFrame(nested_data, nested_schema)

null_dup_data = [
    (1, "A", 10),
    (1, "A", 10),
    (2, None, 20),
    (3, "C", None),
    (4, None, None),
]
null_dup_df = spark.createDataFrame(null_dup_data, ["id", "category", "value"])

left_df = spark.createDataFrame([(1, "x"), (2, "y"), (2, "y"), (3, "z")], ["id", "tag"])
right_df = spark.createDataFrame([(2, "y"), (3, "z"), (4, "w")], ["id", "tag"])

In [3]:
nested_df.show(truncate=False)

+---+--------------------+---------------------------------+-----------+
|id |skills              |profile_map                      |code_struct|
+---+--------------------+---------------------------------+-----------+
|1  |[python, spark, aws]|{laptop -> dell, level -> senior}|{A, 10}    |
|2  |[sql, powerbi]      |{laptop -> hp, level -> mid}     |{B, 20}    |
|3  |[]                  |{laptop -> mac, level -> lead}   |{C, 30}    |
+---+--------------------+---------------------------------+-----------+



In [4]:
print("Employees")
emp_df.show(truncate=False)

print("Departments")
dept_df.show(truncate=False)

print("Bonuses")
bonus_df.show(truncate=False)

print("Nested")
nested_df.show(truncate=False)

print("Null/Duplicates")
null_dup_df.show(truncate=False)

Employees
+------+--------+-------+------+---+---------+----------+----------+------+-----+
|emp_id|emp_name|dept   |salary|age|city     |manager_id|join_date |rating|grade|
+------+--------+-------+------+---+---------+----------+----------+------+-----+
|101   |Anuj    |IT     |90000 |29 |Mumbai   |1001      |2023-01-10|4.7   |NULL |
|102   |Riya    |HR     |60000 |31 |Pune     |1002      |2022-11-05|4.2   |A    |
|103   |Vikas   |IT     |85000 |27 |Bengaluru|1001      |2024-02-12|4.5   |B    |
|104   |Sneha   |Finance|95000 |35 |Mumbai   |1003      |2021-08-19|4.8   |A    |
|105   |Amit    |Sales  |50000 |26 |Delhi    |1004      |2023-06-01|3.9   |NULL |
|106   |Pooja   |HR     |65000 |30 |Chennai  |1002      |2020-12-11|4.1   |B    |
|107   |Karan   |IT     |120000|33 |Hyderabad|1001      |2019-03-14|4.9   |A    |
|108   |Meera   |Sales  |52000 |28 |Pune     |1004      |2024-01-18|3.7   |C    |
|109   |Rohit   |Finance|88000 |32 |Delhi    |1003      |2022-09-09|4.0   |B    |
|110  

In [5]:
emp_df.select("*").show()

+------+--------+-------+------+---+---------+----------+----------+------+-----+
|emp_id|emp_name|   dept|salary|age|     city|manager_id| join_date|rating|grade|
+------+--------+-------+------+---+---------+----------+----------+------+-----+
|   101|    Anuj|     IT| 90000| 29|   Mumbai|      1001|2023-01-10|   4.7| NULL|
|   102|    Riya|     HR| 60000| 31|     Pune|      1002|2022-11-05|   4.2|    A|
|   103|   Vikas|     IT| 85000| 27|Bengaluru|      1001|2024-02-12|   4.5|    B|
|   104|   Sneha|Finance| 95000| 35|   Mumbai|      1003|2021-08-19|   4.8|    A|
|   105|    Amit|  Sales| 50000| 26|    Delhi|      1004|2023-06-01|   3.9| NULL|
|   106|   Pooja|     HR| 65000| 30|  Chennai|      1002|2020-12-11|   4.1|    B|
|   107|   Karan|     IT|120000| 33|Hyderabad|      1001|2019-03-14|   4.9|    A|
|   108|   Meera|  Sales| 52000| 28|     Pune|      1004|2024-01-18|   3.7|    C|
|   109|   Rohit|Finance| 88000| 32|    Delhi|      1003|2022-09-09|   4.0|    B|
|   110|   Nisha

In [7]:
emp_df.select(F.col("emp_name").alias("name"),(F.col("salary")*0.1).alias("bonus")).show()

+-----+-------+
| name|  bonus|
+-----+-------+
| Anuj| 9000.0|
| Riya| 6000.0|
|Vikas| 8500.0|
|Sneha| 9500.0|
| Amit| 5000.0|
|Pooja| 6500.0|
|Karan|12000.0|
|Meera| 5200.0|
|Rohit| 8800.0|
|Nisha| 7000.0|
+-----+-------+



In [18]:
emp_df.select((F.col("emp_id" )*10).alias("id"),"emp_name").show()

+----+--------+
|  id|emp_name|
+----+--------+
|1010|    Anuj|
|1020|    Riya|
|1030|   Vikas|
|1040|   Sneha|
|1050|    Amit|
|1060|   Pooja|
|1070|   Karan|
|1080|   Meera|
|1090|   Rohit|
|1100|   Nisha|
+----+--------+



In [20]:
emp_df.selectExpr(
    "emp_id",
    "emp_name",
    "salary",
    "salary *0.15 as allowance",
    "upper(dept) as dept_upper",
).show()

+------+--------+------+---------+----------+
|emp_id|emp_name|salary|allowance|dept_upper|
+------+--------+------+---------+----------+
|   101|    Anuj| 90000| 13500.00|        IT|
|   102|    Riya| 60000|  9000.00|        HR|
|   103|   Vikas| 85000| 12750.00|        IT|
|   104|   Sneha| 95000| 14250.00|   FINANCE|
|   105|    Amit| 50000|  7500.00|     SALES|
|   106|   Pooja| 65000|  9750.00|        HR|
|   107|   Karan|120000| 18000.00|        IT|
|   108|   Meera| 52000|  7800.00|     SALES|
|   109|   Rohit| 88000| 13200.00|   FINANCE|
|   110|   Nisha| 70000| 10500.00|       OPS|
+------+--------+------+---------+----------+



In [21]:
#withColumn used to create new column or modify the existing one

emp_df.withColumn("salary afer hike",F.col("salary")+5000).show()

+------+--------+-------+------+---+---------+----------+----------+------+-----+----------------+
|emp_id|emp_name|   dept|salary|age|     city|manager_id| join_date|rating|grade|salary afer hike|
+------+--------+-------+------+---+---------+----------+----------+------+-----+----------------+
|   101|    Anuj|     IT| 90000| 29|   Mumbai|      1001|2023-01-10|   4.7| NULL|           95000|
|   102|    Riya|     HR| 60000| 31|     Pune|      1002|2022-11-05|   4.2|    A|           65000|
|   103|   Vikas|     IT| 85000| 27|Bengaluru|      1001|2024-02-12|   4.5|    B|           90000|
|   104|   Sneha|Finance| 95000| 35|   Mumbai|      1003|2021-08-19|   4.8|    A|          100000|
|   105|    Amit|  Sales| 50000| 26|    Delhi|      1004|2023-06-01|   3.9| NULL|           55000|
|   106|   Pooja|     HR| 65000| 30|  Chennai|      1002|2020-12-11|   4.1|    B|           70000|
|   107|   Karan|     IT|120000| 33|Hyderabad|      1001|2019-03-14|   4.9|    A|          125000|
|   108|  

In [23]:
from pyspark.sql.functions import col,lit
emp_df.withColumn("is_mumbai",F.when(F.col("city")=="Mumbai",True).otherwise(False))\
.withColumn("test1",lit("test")).show()

+------+--------+-------+------+---+---------+----------+----------+------+-----+---------+-----+
|emp_id|emp_name|   dept|salary|age|     city|manager_id| join_date|rating|grade|is_mumbai|test1|
+------+--------+-------+------+---+---------+----------+----------+------+-----+---------+-----+
|   101|    Anuj|     IT| 90000| 29|   Mumbai|      1001|2023-01-10|   4.7| NULL|     true| test|
|   102|    Riya|     HR| 60000| 31|     Pune|      1002|2022-11-05|   4.2|    A|    false| test|
|   103|   Vikas|     IT| 85000| 27|Bengaluru|      1001|2024-02-12|   4.5|    B|    false| test|
|   104|   Sneha|Finance| 95000| 35|   Mumbai|      1003|2021-08-19|   4.8|    A|     true| test|
|   105|    Amit|  Sales| 50000| 26|    Delhi|      1004|2023-06-01|   3.9| NULL|    false| test|
|   106|   Pooja|     HR| 65000| 30|  Chennai|      1002|2020-12-11|   4.1|    B|    false| test|
|   107|   Karan|     IT|120000| 33|Hyderabad|      1001|2019-03-14|   4.9|    A|    false| test|
|   108|   Meera|  S

In [25]:
emp_df.withColumnRenamed("emp_name","employee_name").show()

+------+-------------+-------+------+---+---------+----------+----------+------+-----+
|emp_id|employee_name|   dept|salary|age|     city|manager_id| join_date|rating|grade|
+------+-------------+-------+------+---+---------+----------+----------+------+-----+
|   101|         Anuj|     IT| 90000| 29|   Mumbai|      1001|2023-01-10|   4.7| NULL|
|   102|         Riya|     HR| 60000| 31|     Pune|      1002|2022-11-05|   4.2|    A|
|   103|        Vikas|     IT| 85000| 27|Bengaluru|      1001|2024-02-12|   4.5|    B|
|   104|        Sneha|Finance| 95000| 35|   Mumbai|      1003|2021-08-19|   4.8|    A|
|   105|         Amit|  Sales| 50000| 26|    Delhi|      1004|2023-06-01|   3.9| NULL|
|   106|        Pooja|     HR| 65000| 30|  Chennai|      1002|2020-12-11|   4.1|    B|
|   107|        Karan|     IT|120000| 33|Hyderabad|      1001|2019-03-14|   4.9|    A|
|   108|        Meera|  Sales| 52000| 28|     Pune|      1004|2024-01-18|   3.7|    C|
|   109|        Rohit|Finance| 88000| 32|  

In [26]:
df=emp_df.withColumnRenamed("age","Age")
mapping={
    "employee_name":"emp_name",
    "dept":"depart"
}
for old,new in mapping.items():
    df=df.withColumnRenamed(old,new)

In [27]:
df.show()

+------+--------+-------+------+---+---------+----------+----------+------+-----+
|emp_id|emp_name| depart|salary|Age|     city|manager_id| join_date|rating|grade|
+------+--------+-------+------+---+---------+----------+----------+------+-----+
|   101|    Anuj|     IT| 90000| 29|   Mumbai|      1001|2023-01-10|   4.7| NULL|
|   102|    Riya|     HR| 60000| 31|     Pune|      1002|2022-11-05|   4.2|    A|
|   103|   Vikas|     IT| 85000| 27|Bengaluru|      1001|2024-02-12|   4.5|    B|
|   104|   Sneha|Finance| 95000| 35|   Mumbai|      1003|2021-08-19|   4.8|    A|
|   105|    Amit|  Sales| 50000| 26|    Delhi|      1004|2023-06-01|   3.9| NULL|
|   106|   Pooja|     HR| 65000| 30|  Chennai|      1002|2020-12-11|   4.1|    B|
|   107|   Karan|     IT|120000| 33|Hyderabad|      1001|2019-03-14|   4.9|    A|
|   108|   Meera|  Sales| 52000| 28|     Pune|      1004|2024-01-18|   3.7|    C|
|   109|   Rohit|Finance| 88000| 32|    Delhi|      1003|2022-09-09|   4.0|    B|
|   110|   Nisha

In [34]:
#filter and where are used to filter rows

emp_df.filter(F.col("salary")>70000)\
.where((F.col("dept")=="IT") & (F.col("rating") >= 4.5)).show()

+------+--------+----+------+---+---------+----------+----------+------+-----+
|emp_id|emp_name|dept|salary|age|     city|manager_id| join_date|rating|grade|
+------+--------+----+------+---+---------+----------+----------+------+-----+
|   101|    Anuj|  IT| 90000| 29|   Mumbai|      1001|2023-01-10|   4.7| NULL|
|   103|   Vikas|  IT| 85000| 27|Bengaluru|      1001|2024-02-12|   4.5|    B|
|   107|   Karan|  IT|120000| 33|Hyderabad|      1001|2019-03-14|   4.9|    A|
+------+--------+----+------+---+---------+----------+----------+------+-----+



In [35]:
#distinct only returns unique values

left_df.show()

+---+---+
| id|tag|
+---+---+
|  1|  x|
|  2|  y|
|  2|  y|
|  3|  z|
+---+---+



In [37]:
left_df.distinct().show()

+---+---+
| id|tag|
+---+---+
|  1|  x|
|  2|  y|
|  3|  z|
+---+---+



In [38]:
#disintic is entire data frame dropDuplicates is used for selecteed columns as wellas entire

null_dup_df.dropDuplicates().show()
emp_df.select("dept","city").dropDuplicates().show()

+---+--------+-----+
| id|category|value|
+---+--------+-----+
|  1|       A|   10|
|  2|    NULL|   20|
|  4|    NULL| NULL|
|  3|       C| NULL|
+---+--------+-----+

+-------+---------+
|   dept|     city|
+-------+---------+
|     IT|   Mumbai|
|     HR|     Pune|
|Finance|   Mumbai|
|     IT|Bengaluru|
|     HR|  Chennai|
|  Sales|    Delhi|
|  Sales|     Pune|
|     IT|Hyderabad|
|Finance|    Delhi|
|    Ops|   Mumbai|
+-------+---------+



In [39]:
emp_df.select("city").dropDuplicates().show()

+---------+
|     city|
+---------+
|   Mumbai|
|     Pune|
|Bengaluru|
|  Chennai|
|    Delhi|
|Hyderabad|
+---------+



In [41]:
emp_df.limit(3).show()

+------+--------+----+------+---+---------+----------+----------+------+-----+
|emp_id|emp_name|dept|salary|age|     city|manager_id| join_date|rating|grade|
+------+--------+----+------+---+---------+----------+----------+------+-----+
|   101|    Anuj|  IT| 90000| 29|   Mumbai|      1001|2023-01-10|   4.7| NULL|
|   102|    Riya|  HR| 60000| 31|     Pune|      1002|2022-11-05|   4.2|    A|
|   103|   Vikas|  IT| 85000| 27|Bengaluru|      1001|2024-02-12|   4.5|    B|
+------+--------+----+------+---+---------+----------+----------+------+-----+



In [42]:
emp_df.sort("salary").show()
emp_df.orderBy(F.col("salary").desc(),F.col("age").asc()).show()

+------+--------+-------+------+---+---------+----------+----------+------+-----+
|emp_id|emp_name|   dept|salary|age|     city|manager_id| join_date|rating|grade|
+------+--------+-------+------+---+---------+----------+----------+------+-----+
|   105|    Amit|  Sales| 50000| 26|    Delhi|      1004|2023-06-01|   3.9| NULL|
|   108|   Meera|  Sales| 52000| 28|     Pune|      1004|2024-01-18|   3.7|    C|
|   102|    Riya|     HR| 60000| 31|     Pune|      1002|2022-11-05|   4.2|    A|
|   106|   Pooja|     HR| 65000| 30|  Chennai|      1002|2020-12-11|   4.1|    B|
|   110|   Nisha|    Ops| 70000| 29|   Mumbai|      1005|2023-04-22|   4.3| NULL|
|   103|   Vikas|     IT| 85000| 27|Bengaluru|      1001|2024-02-12|   4.5|    B|
|   109|   Rohit|Finance| 88000| 32|    Delhi|      1003|2022-09-09|   4.0|    B|
|   101|    Anuj|     IT| 90000| 29|   Mumbai|      1001|2023-01-10|   4.7| NULL|
|   104|   Sneha|Finance| 95000| 35|   Mumbai|      1003|2021-08-19|   4.8|    A|
|   107|   Karan